# CPE342 Task 4: GPU training on Colab (round 37, DINOv2-L: fold 0 + gate, then folds 1-4 + full refit)

Runs the same trainer as the laptop (`scripts/train_task4.py`, identical recipe and folds) on a bigger GPU. Training images only; no
test image or test statistic enters any decision (the full refit only *predicts* the test images, as on the laptop).

**Before running**
1. On the laptop: `python scripts/make_colab_bundle.py` → `colab/upload/cpe342_code.zip` (≈4 MB) and `colab/upload/t4_data.zip`
   (≈0.6 GB). Upload both to Google Drive, folder `MyDrive/cpe342_colab/`. Keep the folder private (competition data).
2. Colab: *Runtime → Change runtime type → A100* (or L4 / H100). A T4 has no bf16 and is slower than the laptop: do not use it.
3. *Runtime → Run all*. Keep the browser tab open (Colab Pro stops when the tab closes or after long idle).

**Resuming:** everything is written to `MyDrive/cpe342_colab/out/` as soon as it exists, and finished folds are skipped. If the
session ends, reconnect and *Run all* again; it continues with the next unfinished step.

**Gate (fixed before the run, as the laptop queue):** after fold 0 its five views are scored against v23 on fold 0
(`fc_r31_t4_v23_eval.py`); folds 1-4 and the full refit run only if at least one fixed-weight variant passes the hard-view rule AND
the v24 check (zoom70, HUD hidden and real dark / low-contrast frames > 0, full >= -0.0005). Otherwise the notebook stops.

**When it is done:** copy from `MyDrive/cpe342_colab/out/` to the laptop repo:
`preds/t4_dinov2L_fold{0-4}.npz`, `preds/t4_dinov2L_full.npz` → `outputs/preds/`;
`r30_views/dinov2L_fold{0-4}.npz` → `experiments/claude/outputs/r30_views/`; `logs/*dinov2L*` → `outputs/logs/`.
The model checkpoints (`models/`, 1.2 GB each) are only needed to recompute views and can stay on Drive.

In [ ]:
# ---- settings -------------------------------------------------------------------------------------------------------------
TAG = "dinov2L"
FOLDS = [0, 1, 2, 3, 4]       # fold 0 first; folds 1-4 only if fold 0 passes the gate
GATE = True                   # False: train every fold without the fold-0 gate
DO_FULL = True                # full Train+Val refit -> test predictions (the member's test input)
DRIVE_DIR = "/content/drive/MyDrive/cpe342_colab"
MODELS_ON_DRIVE = True        # 1.2 GB per checkpoint (about 6 GB in all); False keeps them on the Colab disk (lost with the session)
GRAD_CKPT = False             # gradient checkpointing changes memory only, not the result; a 24-80 GB GPU does not need it
ACCUM = 2                     # 64 images per optimiser step as 2 x 32, exactly as the laptop's fold 0 (keeps the recipe identical)
RECIPE = ["--model", "vit_large_patch14_reg4_dinov2.lvd142m", "--h", "126", "--w", "238", "--bs", "64", "--accum", str(ACCUM),
          "--lr", "3e-5", "--layer-decay", "0.85", "--warmup-epochs", "2", "--epochs", "15", "--p-hud", "0.5"]
RECIPE += ["--grad-ckpt"] if GRAD_CKPT else []

In [ ]:
# ---- GPU check ------------------------------------------------------------------------------------------------------------
import subprocess
import torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
cap = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), "compute capability", cap, "| torch", torch.__version__)
assert cap[0] >= 8, "this GPU has no native bf16 (T4?): change the runtime type to A100, L4 or H100"

In [ ]:
# ---- Drive, code and data --------------------------------------------------------------------------------------------------
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
D, R = Path(DRIVE_DIR), Path("/content/cpe342")
for z in ("cpe342_code.zip", "t4_data.zip"):
    assert (D / z).exists(), f"upload {z} to {DRIVE_DIR}"
if not (R / "src" / "task4.py").exists():
    zipfile.ZipFile(D / "cpe342_code.zip").extractall(R)
DATA = R / "04_image_classification/_download/task4_image_classification"
if not (DATA / "test.csv").exists() or len(os.listdir(DATA / "test")) < 25889:
    shutil.copy(D / "t4_data.zip", "/content/t4_data.zip")          # local copy first: unzipping straight from Drive is slow
    zipfile.ZipFile("/content/t4_data.zip").extractall(R)
    os.remove("/content/t4_data.zip")
print(len(os.listdir(DATA / "train")), "training images,", len(os.listdir(DATA / "test")), "test images")

# outputs live on Drive, so they survive the session; the fold indices from the bundle are copied over first
persist = {"outputs/preds": "preds", "outputs/logs": "logs", "experiments/claude/outputs/r30_views": "r30_views"}
if MODELS_ON_DRIVE:
    persist["outputs/models"] = "models"
for local, remote in persist.items():
    src_dir, dst = R / local, D / "out" / remote
    dst.mkdir(parents=True, exist_ok=True)
    if src_dir.is_symlink():
        continue
    if src_dir.exists():
        for f in src_dir.iterdir():
            if not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
        shutil.rmtree(src_dir)
    src_dir.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(dst, src_dir)
print({k: sorted(p.name for p in (D / "out" / v).iterdir())[:8] for k, v in persist.items()})

In [ ]:
# ---- packages (timm as on the laptop) --------------------------------------------------------------------------------------
!pip -q install timm==1.0.30
import numpy, pandas, sklearn, timm, torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__, "| sklearn", sklearn.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)

In [ ]:
# ---- 1-epoch benchmark on 1,408 training images (also downloads the DINOv2-L weights, 1.2 GB) ---------------------------
import sys
import time
import numpy as np
os.chdir(R)
sys.path.insert(0, str(R))
from src import task4

tr_df, _ = task4.load_frames()
idx = np.arange(1408)
arr = task4.load_uint8(list(tr_df.path.values[idx]))
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
m, h = task4.train_fold("vit_large_patch14_reg4_dinov2.lvd142m", arr, tr_df.label.values[idx], np.arange(1280), np.arange(1280, 1408),
                        img_size=(126, 238), epochs=1, bs=64, lr=3e-5, layer_decay=0.85, aug_kw={"p_hud": 0.5}, grad_ckpt=GRAD_CKPT,
                        warmup_epochs=1, accum=ACCUM)
per_img = h[-1]["secs"] / 1280
print(f"peak GPU memory {torch.cuda.max_memory_reserved() / 2**30:.1f} GiB; {per_img * 1000:.1f} ms per training image (first epoch, "
      f"includes warm-up) -> about {per_img * 25237 / 60:.0f} min per fold epoch, {per_img * 25237 * 15 / 3600:.1f} h per fold "
      f"(laptop: 656 s per epoch, 2.7 h per fold), {per_img * 25237 * 15 * (len(FOLDS) + 1.25 * DO_FULL) / 3600:.1f} h for this run")
del m, arr
torch.cuda.empty_cache()

In [ ]:
# ---- training: folds, their views, then the full refit -----------------------------------------------------------------------
import subprocess

def run(args, extra_env=None):
    print("$ python", " ".join(args), flush=True)
    p = subprocess.Popen(["python", "-u", *args], cwd=R, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONIOENCODING": "utf-8", **(extra_env or {})})
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    print(f"exit {rc} [{time.strftime('%H:%M')}]", flush=True)
    return rc

def gate_passed():
    # v23-based scores of the folds with views (fc_r31_t4_v23_eval.py); PASS if one variant passes the hard-view rule and the v24 check
    out = subprocess.run(["python", "-u", "experiments/claude/fc_r31_t4_v23_eval.py", TAG], cwd=R, capture_output=True, text=True,
                         env={**os.environ, "PYTHONIOENCODING": "utf-8"}).stdout
    (D / "out" / "logs" / f"r37_fc_{TAG}_fold0_colab.log").write_text(out, encoding="utf-8")
    print("\n".join(l for l in out.splitlines() if "Warning" not in l))
    return any("hard-view rule PASS" in l and "v24 check (beyond the grid) PASS" in l for l in out.splitlines())

t_start = time.time()
for f in FOLDS:
    if not (R / f"outputs/preds/t4_{TAG}_fold{f}.npz").exists():
        assert run(["scripts/train_task4.py", "--tag", TAG, *RECIPE, "--folds", str(f)]) == 0, f"fold {f} failed"
    if not (R / f"experiments/claude/outputs/r30_views/{TAG}_fold{f}.npz").exists():
        # the five scoring views of this fold's validation images (full, zoom70, HUD hidden, gamma 3.5, 75% grid)
        assert run(["experiments/claude/em_r30_t4_hardviews.py", TAG], {"VIEWS_ONLY": "1"}) == 0, f"views of fold {f} failed"
    if f == 0 and GATE:
        ok = gate_passed()
        print(f"\nfold-0 gate: {'PASS -> folds 1-4 and the full refit' if ok else 'FAIL -> stop (no further compute units spent)'}")
        if not ok:
            break
else:
    if DO_FULL and not (R / f"outputs/preds/t4_{TAG}_full.npz").exists():
        assert run(["scripts/train_task4.py", "--tag", TAG, *RECIPE, "--full"]) == 0, "full refit failed"
print(f"done in {(time.time() - t_start) / 3600:.2f} h")

In [ ]:
# ---- what to copy back to the laptop -------------------------------------------------------------------------------------------
for sub in ("preds", "r30_views", "logs"):
    for f in sorted((D / "out" / sub).glob(f"*{TAG}*")):
        print(f"{f}  {f.stat().st_size / 1e6:.1f} MB")